# Qwen3-Reranker-0.6B — DIMER query-document reranking tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/qwen3-reranker-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/qwen3-reranker-pipeline/blob/main/tutorials/qwen3_reranker_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Qwen%2FQwen3--Reranker--0.6B-ffcc4d?style=flat)](https://huggingface.co/Qwen/Qwen3-Reranker-0.6B) [![Upstream](https://img.shields.io/badge/Upstream-QwenLM%2FQwen3--Embedding-181717?style=flat&logo=github&logoColor=white)](https://github.com/QwenLM/Qwen3-Embedding) [![arXiv](https://img.shields.io/badge/arXiv-2506.05176-b31b1b.svg)](https://arxiv.org/abs/2506.05176)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** pointwise query-document relevance reranking using the pinned Qwen3-Reranker-0.6B weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/qwen3_reranker_pipeline/pipeline.py` at revision `90e72d3b093a`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `e61197ed45024b0ed8a2d74b80b4d909f1255473` (~1207 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/1); edit the repository and regenerate rather than editing cells.

At inference each `(query, document)` pair is wrapped in the fixed upstream system/user/assistant prompt together with an instruction, one forward pass of the causal language model reads the last-position logits of the `yes` and `no` tokens, and a two-way softmax turns them into a **relevance score** in [0, 1]: the `yes` share. **The score is not a calibrated probability**, no threshold is shipped, and the `ranking` the pipeline returns is an ordering of the supplied pairs, not an acceptance decision — the caller owns any cut-off. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. What the upstream checkpoint supplies is the model, tokenizer and prompt convention; what the carried pipeline module adds is manifest verification, input validation and ceilings, prompt assembly, the two-logit read-out, a fixed output contract and the `validate_inputs` and `evaluation_report` stage helpers. Free-text generation is deliberately not reachable through this package.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, author a synthetic query and candidate set (or upload your own), surface the pipeline's ceilings, stage and digest-verify the immutable upstream snapshot, validate the pairs into an input manifest, rerank through the public API, read scores and ranking correctly, understand from the machine-readable evaluation report why no metric is reported and what labelled judgements a real evaluation needs, and export the ranking with identifiers plus provenance.

**This notebook does not demonstrate:** embedding or retrieval over a corpus (the Qwen3-Embedding sibling covers first-stage retrieval), text generation, listwise or pairwise comparison between documents, multilingual quality claims, or any calibrated relevance threshold. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (bfloat16 there, the checkpoint's native dtype); the model card's CPU smoke scored four short pairs in about 1 s after a 5 s load, so the three-pair default runs in seconds on a hosted CPU runtime. The pinned `torch==2.14.0` install and the 1.19 GB checkpoint are the largest downloads of the run.
- **Knowledge:** basic Python; what a softmax over two logits is and why it is not a calibrated probability; what a relevance judgement is.
- **Data:** the default sample is a synthetic query and three candidate passages authored in code, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one UTF-8 text file whose first non-empty line is the query and whose remaining non-empty lines are candidate documents (one per line, at most 32 documents, each at most 100,000 characters). Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded text remains in the notebook runtime; this pipeline does not send it to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `Qwen/Qwen3-Reranker-0.6B` snapshot (~1207 MB) at revision `e61197ed4502…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'qwen3-reranker-pipeline',
    'repository_revision': '90e72d3b093a349775a28b2314147260d62e885d',
    'embedded_module': 'src/qwen3_reranker_pipeline/pipeline.py',
    'module_sha256': '0a311ac03099a38acf6216d2203e9ce7eb013e8e45a985f6001a957803db6546',
    'generator': 'build_notebook.py/1',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/qwen3_reranker_pipeline/pipeline.py` @ `90e72d3b093a`)

This cell **is** the repository's pipeline module: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the module's, byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (currently 1: the default weights directory becomes working-directory-relative because a notebook has no `__file__`). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever this cell and the module diverge, so what you run here is what the repository tests. Nothing in this cell runs a model yet.

In [ ]:
from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

MODEL_ID = "Qwen/Qwen3-Reranker-0.6B"
MODEL_REVISION = "e61197ed45024b0ed8a2d74b80b4d909f1255473"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "qwen3-reranker-0.6b"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Prompt contract from the pinned upstream README ("Using Transformers"): a fixed system prompt, the
# instruction/query/document block, and the assistant prefix with an empty <think> block; the score is
# the softmax over the "no"/"yes" logits at the last position. Token ids are pinned by the snapshot's
# 1_LogitScore/config.json and cross-checked against the tokenizer at load time.
PREFIX = (
    "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the "
    'Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n'
)
SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
DEFAULT_INSTRUCTION = "Given a web search query, retrieve relevant passages that answer the query"
YES_TOKEN, NO_TOKEN = "yes", "no"
YES_TOKEN_ID, NO_TOKEN_ID = 9693, 2152
MAX_TEXT_TOKENS = 8192  # total prompt length incl. prefix/suffix; the pair is truncated longest-first to fit
MAX_TEXT_CHARS = 100_000  # per query or document, pre-tokenisation guard
MAX_PAIRS = 32  # (query, document) pairs per rerank() call
SCORE_KIND = "relevance score, not a calibrated probability"


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = hashlib.sha256()
        with open(file_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                digest.update(chunk)
        if digest.hexdigest() != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}")
    return manifest


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def format_pair(query: str, document: str, instruction: str = DEFAULT_INSTRUCTION) -> str:
    """Upstream `format_instruction`: the user-turn body between PREFIX and SUFFIX."""
    return f"<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {document}"


INPUT_SCHEMA: dict[str, Any] = {
    "input": "sequence of (query, document) pairs of two non-empty str; one score is returned per pair",
    "pairs": [1, MAX_PAIRS],
    "text_chars": [1, MAX_TEXT_CHARS],
    "prompt_tokens": [1, MAX_TEXT_TOKENS],
    "score_range": [0.0, 1.0],
    "preprocessing": (
        "each pair becomes '<Instruct>: <instruction>\\n<Query>: …\\n<Document>: …' between the fixed "
        "upstream PREFIX and SUFFIX, truncated longest-first to fit MAX_TEXT_TOKENS; the score is the "
        "two-way softmax share of the yes logit against the no logit at the last position"
    ),
}


def _check_inputs(pairs: Any, instruction: str) -> list[tuple[str, str]]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the pairs as a list."""
    if isinstance(pairs, str | bytes) or not isinstance(pairs, Sequence):
        raise TypeError("pairs must be a list of (query, document) pairs")
    if not 1 <= len(pairs) <= MAX_PAIRS:
        raise ValueError(f"pairs must hold 1..{MAX_PAIRS} items, got {len(pairs)}")
    for i, pair in enumerate(pairs):
        if isinstance(pair, str | bytes) or not isinstance(pair, Sequence) or len(pair) != 2:
            raise TypeError(f"pairs[{i}] must be a (query, document) pair of two str")
        for name, text in zip(("query", "document"), pair, strict=True):
            if not isinstance(text, str):
                raise TypeError(f"pairs[{i}] {name} must be str, got {type(text).__name__}")
            if not text.strip():
                raise ValueError(f"pairs[{i}] {name} is empty")
            if len(text) > MAX_TEXT_CHARS:
                raise ValueError(f"pairs[{i}] {name} has {len(text)} chars; ceiling is {MAX_TEXT_CHARS}")
    if not isinstance(instruction, str) or not instruction.strip():
        raise ValueError("instruction must be a non-empty str")
    return [(pair[0], pair[1]) for pair in pairs]


def validate_inputs(
    pairs: Sequence[Sequence[str]],
    instruction: str = DEFAULT_INSTRUCTION,
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-pair observations, verdict).

    Rejection is reported by raising exactly as ``rerank`` would — both route through
    ``_check_inputs``. Prompt-level truncation cannot be observed here because it happens inside
    the tokenizer; ``rerank`` reports it in ``truncated``.
    """
    checked = _check_inputs(pairs, instruction)
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per pair")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[i] if names else f"pair-{i}",
                "query_chars": len(query),
                "document_chars": len(document),
            }
            for i, (query, document) in enumerate(checked)
        ],
        "n_pairs": len(checked),
        "instruction": instruction,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], judgements: Sequence[Any] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even though no metric exists here.

    The repository ships no ranking-metric helper, so the verdict is always ``not-measurable``
    (EVAL9), including when ``judgements`` is supplied: the parameter exists for interface parity
    with the fleet's other pipelines and is recorded in ``reason`` rather than scored. Inventing
    nDCG or MRR here would hide the fact that a real evaluation needs judged candidates over many
    queries and the caller's own metric code.
    """
    scores = result["scores"]
    supplied = judgements is not None
    return {
        "task": "pointwise query-document relevance reranking",
        "score_semantics": (
            f"{SCORE_KIND}: the softmax share of the yes logit against the no logit, in [0, 1]; it "
            "orders candidates for one query, is not comparable as an absolute value across queries "
            "or instructions, and carries no shipped acceptance threshold"
        ),
        "sample_kind": sample_kind,
        "n_pairs": len(scores),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": (
            "ranking quality needs relevance judgements and the repository ships no metric helper"
            + (
                "; judgements were supplied but no metric helper exists to score them here"
                if supplied
                else "; the evaluated sample carries none"
            )
        ),
        "needs": (
            "per-query relevance judgements (binary or graded) over enough queries to state a "
            "dispersion, scored with the caller's own nDCG@k, MRR or precision@k code; a single "
            "query's ordering is a plumbing check, not a retrieval measurement"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class Qwen3RerankerPipeline:
    """Pointwise reranker. `_runner` maps pair bodies to ([no, yes] last-position logits, token counts)."""

    _runner: Callable[[list[str]], tuple[np.ndarray, list[int]]]
    device: str

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Qwen3RerankerPipeline:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        dtype = torch.bfloat16 if resolved_device.startswith("cuda") else torch.float32
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), dict(local_files_only=True)
        elif allow_download:
            source, kwargs = MODEL_ID, dict(revision=MODEL_REVISION)
        else:
            raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        tokenizer = AutoTokenizer.from_pretrained(
            source, padding_side="left", trust_remote_code=False, **kwargs
        )
        ids = (tokenizer.convert_tokens_to_ids(YES_TOKEN), tokenizer.convert_tokens_to_ids(NO_TOKEN))
        if ids != (YES_TOKEN_ID, NO_TOKEN_ID):
            raise RuntimeError(f"tokenizer maps yes/no to {ids}, expected {(YES_TOKEN_ID, NO_TOKEN_ID)}")
        model = AutoModelForCausalLM.from_pretrained(source, dtype=dtype, trust_remote_code=False, **kwargs)
        model = model.to(resolved_device).eval()
        prefix_ids = tokenizer.encode(PREFIX, add_special_tokens=False)
        suffix_ids = tokenizer.encode(SUFFIX, add_special_tokens=False)
        body_budget = MAX_TEXT_TOKENS - len(prefix_ids) - len(suffix_ids)

        def runner(bodies: list[str]) -> tuple[np.ndarray, list[int]]:
            enc = tokenizer(
                bodies,
                padding=False,
                truncation="longest_first",
                return_attention_mask=False,
                max_length=body_budget,
            )
            enc["input_ids"] = [prefix_ids + row + suffix_ids for row in enc["input_ids"]]
            batch = tokenizer.pad(enc, padding=True, return_tensors="pt")
            batch = batch.to(resolved_device)
            with torch.inference_mode():
                last = model(**batch).logits[:, -1, :]
            pair_logits = torch.stack([last[:, NO_TOKEN_ID], last[:, YES_TOKEN_ID]], dim=1)
            counts = batch["attention_mask"].sum(dim=1).tolist()
            return pair_logits.float().cpu().numpy(), [int(c) for c in counts]

        return cls(runner, resolved_device)

    def _validate(self, pairs: Any, instruction: str) -> list[tuple[str, str]]:
        return _check_inputs(pairs, instruction)

    def rerank(
        self,
        pairs: Sequence[Sequence[str]],
        instruction: str = DEFAULT_INSTRUCTION,
    ) -> dict[str, Any]:
        """Score up to MAX_PAIRS (query, document) pairs; `scores` align with `pairs`; no threshold."""
        pairs = self._validate(pairs, instruction)
        bodies = [format_pair(q, d, instruction) for q, d in pairs]
        logits, n_tokens = self._runner(bodies)
        logits = np.asarray(logits, dtype=np.float64)
        if logits.shape != (len(pairs), 2):
            raise RuntimeError(f"backend returned {logits.shape}, expected ({len(pairs)}, 2)")
        shifted = logits - logits.max(axis=1, keepdims=True)
        probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
        scores = probs[:, 1]
        return {
            "scores": [float(s) for s in scores],
            "ranking": [int(i) for i in np.argsort(-scores, kind="stable")],
            "score_kind": SCORE_KIND,
            "instruction": instruction,
            "n_tokens": list(n_tokens),
            "truncated": [n >= MAX_TEXT_TOKENS for n in n_tokens],
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `13`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `e61197ed4502…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Qwen3RerankerPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "qwen3-reranker-0.6b",
  "modelId": "Qwen/Qwen3-Reranker-0.6B",
  "revision": "e61197ed45024b0ed8a2d74b80b4d909f1255473",
  "files": [
    {
      "path": "1_LogitScore/config.json",
      "bytes": 57,
      "sha256": "73e3156450564d8a98b7e47bcf5aace0f29600828b51937da545571e84db3ff3"
    },
    {
      "path": "README.md",
      "bytes": 14742,
      "sha256": "5bba8c734f6dd3ae48317b4139317e45a7fce48fc55e15670b23a0dd15492ab6"
    },
    {
      "path": "chat_template.jinja",
      "bytes": 741,
      "sha256": "6f682162495ec5b39fd9005c01b6aa2a74669379fe967039f1e2cbbe8752369d"
    },
    {
      "path": "config.json",
      "bytes": 727,
      "sha256": "d479c427a9ca5295218063d4f9aca4f297ab4ac27487cca7af42c84643d51ef0"
    },
    {
      "path": "config_sentence_transformers.json",
      "bytes": 325,
      "sha256": "6a153d6696f78fd588c1c728967f0b773ea869d3c6028f151ce71ebe49140762"
    },
    {
      "path": "generation_config.json",
      "bytes": 214,
      "sha256": "81051cd3f6e77013827148d0b8a6ead93f8ac390d5ab805f849199f0af6a08db"
    },
    {
      "path": "merges.txt",
      "bytes": 1671853,
      "sha256": "8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5"
    },
    {
      "path": "model.safetensors",
      "bytes": 1191588280,
      "sha256": "27cd75a405b9c1b46b59abfd88aaa209e6fed2a1972cde9b70e7659537c5e65b"
    },
    {
      "path": "modules.json",
      "bytes": 280,
      "sha256": "6f13b6b4a89e577b591b2077bca40c67c26541a6740a8809267cb474f90806a9"
    },
    {
      "path": "sentence_bert_config.json",
      "bytes": 362,
      "sha256": "3234ebd224d492cbe8d55d5ec80a3f408451c4db3005bafb64fe1c51c763e01e"
    },
    {
      "path": "tokenizer.json",
      "bytes": 11422654,
      "sha256": "aeb13307a71acd8fe81861d94ad54ab689df773318809eed3cbe794b4492dae4"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 9706,
      "sha256": "253153d0738ceb4c668d2eff957714dd2bea0b56de772a9fdccd96cbf517e6a0"
    },
    {
      "path": "vocab.json",
      "bytes": 2776833,
      "sha256": "ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910"
    }
  ],
  "totalBytes": 1207486774
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Qwen3RerankerPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Author the synthetic sample or optional BYOD

The default sample is **synthetic**: one query and three candidate passages written in this cell — one that answers the query, one on the same topic that does not answer it, and one unrelated — so it needs no download and contains no personal data. It ships **no relevance judgements** beyond the author's intent, so the scores it produces are smoke/sanity evidence that the code path works (the answering passage is expected to outrank the unrelated one), never a retrieval-quality measurement and never benchmark evidence.

BYOD is optional and disabled by default. Expected BYOD input: one UTF-8 text file whose first non-empty line is the query and whose remaining non-empty lines are candidate documents; every document is paired with the query. The upload stays inside this runtime. If you also hold relevance judgements for your candidates, keep them outside the notebook — Section 7 explains what to compute with them.

In [ ]:
import hashlib
import io

USE_BYOD = False  # @param {type:"boolean"}

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    lines = [line.strip() for line in io.StringIO(uploaded[sample_name].decode('utf-8')) if line.strip()]
    if len(lines) < 2:
        raise ValueError(f'{sample_name}: expected a query line followed by at least one document line')
    query, documents = lines[0], lines[1:]
    sample_kind = 'BYOD upload'
else:
    query = 'What is the capital of China?'
    documents = [
        'The capital of China is Beijing, which has been the seat of government since 1949.',
        'Shanghai is the largest city in China by population and a major financial centre.',
        'Gravity is the force by which a planet or other body draws objects toward its centre.',
    ]
    sample_name = 'synthetic_capital_query'
    sample_kind = 'synthetic (authored in this cell)'
doc_ids = [f'doc{index:02d}' for index in range(len(documents))]
pairs = [(query, document) for document in documents]
sample_sha256 = hashlib.sha256('\n'.join([query, *documents]).encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'query': query, 'documents': len(documents), 'text_sha256': sample_sha256})
for doc_id, document in zip(doc_ids, documents, strict=True):
    print(f'{doc_id}: {document[:100]}')

## 5. Validate the pairs → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `rerank` applies — both route through the same private `_check_inputs` — so the pair count 1..`MAX_PAIRS`, the pair shape, non-empty query and document, the character ceiling `MAX_TEXT_CHARS` and a non-empty instruction are enforced identically. It returns an **input manifest** naming the schema and ceilings, each pair's identifier and character counts, the instruction in force, and the verdict; the manifest is written to `outputs/qwen3_reranker_input_manifest.json`. The ceilings are surfaced before any model work: `MAX_PAIRS` (pairs per `rerank` call), `MAX_TEXT_CHARS` (characters per query or document, checked before tokenisation) and `MAX_TEXT_TOKENS` (total prompt tokens including the fixed prefix/suffix; longer prompts are truncated longest-first and flagged per pair in `truncated`). The default `instruction` is printed because it is part of the prompt and changes the scores, so a deployment must fix it deliberately. To show what rejection looks like, the cell also validates a pair with an empty document and records the pipeline's own error message as a finding. The notebook never trims or alters the texts; any token-level truncation happens inside the pipeline and is reported after the call.

In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
ceilings = {'MAX_PAIRS': MAX_PAIRS, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS}
instruction = DEFAULT_INSTRUCTION
print(ceilings)
print({'instruction': instruction})
input_manifest = validate_inputs(pairs, instruction, names=doc_ids)
# Demonstrate rejection on an input that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs([(query, '   ')], instruction)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'empty-document-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/qwen3_reranker_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Rerank and interpret the scores

`rerank(pairs, instruction=…)` returns `scores` (one float in [0, 1] per pair, **aligned with the input order**), `ranking` (pair indices sorted by descending score, stable), `score_kind`, the `instruction` used, `n_tokens` per prompt, `truncated` flags, and the model identity. **Score semantics:** each score is the softmax share of the `yes` logit against the `no` logit — a relevance score that orders candidates for one query; it is not a calibrated probability that the document is relevant, scores for different queries are not comparable as absolute values, and the pipeline ships no threshold. Any accept/reject cut-off (for example "show only candidates above 0.5") is owned by the caller and must be set on their own labelled pairs. Scores differ slightly between the CPU float32 and CUDA bfloat16 paths and can reorder near-tied candidates. The runtime figure is measured on the runtime identified in Section 1 for this batch and includes the first-call warm-up.

In [ ]:
import time

started = time.perf_counter()
result = pipe.rerank(pairs, instruction=instruction)
elapsed = time.perf_counter() - started
scores = result['scores']
checks = {
    'one_score_per_pair': len(scores) == len(pairs),
    'scores_in_unit_interval': all(0.0 <= s <= 1.0 for s in scores),
    'ranking_is_permutation': sorted(result['ranking']) == list(range(len(pairs))),
    'nothing_truncated': not any(result['truncated']),
}
if not all(checks.values()):
    raise RuntimeError(f'rerank output failed a sanity check: {checks}')
print({key: value for key, value in result.items() if key not in ('scores', 'ranking')})
print({'seconds': round(elapsed, 3), 'checks': checks})
print(f'query: {query}')
for rank, index in enumerate(result['ranking'], start=1):
    print(f"{rank:>2}. {doc_ids[index]}  score {scores[index]:.4f}  tokens {result['n_tokens'][index]:>4}  {documents[index][:80]}")

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report — even, as here, when nothing is measurable. The repository ships **no metric helper and reports no performance measure**, so the verdict is always `not-measurable` and the report states what would make the task measurable: per-query relevance judgements (binary or graded) over enough queries to state a dispersion, scored with the caller's own nDCG@k, MRR or precision@k code. Supplying judgements does not change the verdict, because there is no metric helper to score them with; the helper records that fact in `reason` rather than inventing a number, and the upstream benchmark figures quoted in the model card remain upstream claims, not measurements made here. On the synthetic default sample the cell also prints one falsifiable plumbing check — the answering passage should outrank the unrelated one — which is a check on one query, not a retrieval result. The report is written to `outputs/qwen3_reranker_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, sample_kind=sample_kind)
with open('outputs/qwen3_reranker_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
sanity = {}
if not USE_BYOD:
    sanity = {'answering_outranks_unrelated': scores[0] > scores[2]}
    print({'sanity_check': sanity, 'note': 'falsifiable plumbing check on one synthetic query; not a metric'})
if report['verdict'] == 'not-measurable':
    print('No metric is reported: the sample has no relevance judgements and the repository ships no metric helper; compute nDCG/MRR on your own judged pairs.')

## 8. Export the ranking and provenance

The ranking is written as CSV (`outputs/qwen3_reranker_ranking.csv`) with explicit `rank`, `id`, `score`, `n_tokens`, `truncated` and `document` columns, so the ordering survives downstream use. Machine-readable JSON preserves an `items` list with, per pair, its identifier, document text, score, rank, token count and truncation flag (so every score maps back to its input), the query, the instruction, the `ranking`, the `score_kind`, the sanity checks, the ceilings in force, the input manifest, the evaluation report, the sample identity and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, the verified snapshot summary, and the runtime identity (Python, `torch`, `transformers`, device, dtype). No credentials are involved in any step, so none can reach the export.

In [ ]:
import csv

rank_of = {index: rank for rank, index in enumerate(result['ranking'], start=1)}
with open('outputs/qwen3_reranker_ranking.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['rank', 'id', 'score', 'n_tokens', 'truncated', 'document'])
    for index in result['ranking']:
        writer.writerow([rank_of[index], doc_ids[index], f'{scores[index]:.6f}', result['n_tokens'][index], result['truncated'][index], documents[index]])
payload = {
    'query': query,
    'instruction': result['instruction'],
    'items': [
        {'id': doc_ids[index], 'document': documents[index], 'score': scores[index], 'rank': rank_of[index], 'n_tokens': result['n_tokens'][index], 'truncated': result['truncated'][index]}
        for index in range(len(pairs))
    ],
    'ranking': [doc_ids[index] for index in result['ranking']],
    'score_kind': result['score_kind'],
    'sanity_checks': checks,
    'plumbing_check': sanity,
    'ceilings': ceilings,
    'ranking_file': 'outputs/qwen3_reranker_ranking.csv',
    'input_manifest': input_manifest,
    'evaluation_report': report,
    'sample': {'name': sample_name, 'kind': sample_kind, 'text_sha256': sample_sha256},
    'seconds': round(elapsed, 3),
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes')},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'dtype': 'bfloat16' if pipe.device.startswith('cuda') else 'float32',
    },
}
with open('outputs/qwen3_reranker_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

Each score is the `yes` share of a two-way softmax over the reranker's last-position logits: a relevance score that orders candidates for one query, not a calibrated probability, not comparable as an absolute value across queries or instructions, and never converted to a decision by the pipeline — the caller owns any threshold and must set it on their own judged pairs. On the synthetic sample the ordering is plumbing evidence only; the evaluation report is `not-measurable` because no metric can be computed without relevance judgements, and a real evaluation needs judged candidates for many queries and the caller's own nDCG/MRR code. The instruction is part of the prompt and changes the scores; prompts beyond `MAX_TEXT_TOKENS` are truncated longest-first and flagged; the pipeline exposes no generation, no listwise comparison, and no corpus retrieval. The forward pass is deterministic on a fixed device and dtype, but CPU (float32) and CUDA (bfloat16) scores differ slightly and can reorder near-tied candidates.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model snapshot, validate the demonstrated pairs against the enforced ceilings, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, retrieval quality on any domain, a usable relevance threshold, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file is incomplete or altered — delete it from `weights/qwen3-reranker-0.6b/` and rerun Section 3. A `ValueError` naming `MAX_PAIRS` or `MAX_TEXT_CHARS` in Section 5: reduce or shorten the BYOD lines and rerun from Section 4. A `truncated` flag set to `True` in Section 6: that prompt exceeded `MAX_TEXT_TOKENS` and was cut longest-first — shorten the document if the cut matters.

**Next experiments.** Upload a query with a dozen candidates you can judge yourself and compare the pipeline's ranking with your judgements — that is exactly the labelled data the evaluation report asks for; change `instruction` in Section 5 to a task-specific one and observe how the scores move; run the same batch on a CUDA runtime and compare the bfloat16 scores with the CPU float32 ones. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: https://github.com/kurtvalcorza/qwen3-reranker-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/qwen3-reranker-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/qwen3-reranker-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/Qwen/Qwen3-Reranker-0.6B
- Upstream code: https://github.com/QwenLM/Qwen3-Embedding
- Qwen3 Embedding technical report: https://arxiv.org/abs/2506.05176